### This file is corresponding to Construction 1 in our paper that is the RLWR MKFHE based on the FHE scheme  from (https://eprint.iacr.org/2024/960)


In [1]:
## Multiplication functions

In [2]:
import numpy as np
import math
from scipy import signal
from random import SystemRandom
from math import log2, ceil
from time import time
import scipy


"""
poly_multiplication.py implements a polynomial multiplication leveraging scipy's complex FFT implementation.
To multiply degree N polynomials f and g with integer coefficients one calls fast_poly_mult(f,g).
More details on how to use the complex FFT to multiply integer polynomials are given 
https://github.com/rtitiu/polymul-approx-ffts.
"""

def mod_vec(x_vec, modulus):
	x_vec = [int(i) for i in x_vec]
# 	print("x_vec: ", x_vec)
# 	print("modulus: ", modulus)
	log_modulus = ceil(log2(modulus))
	x_vec = np.array(x_vec, dtype = object)
	positive_r = np.bitwise_and(x_vec, modulus - 1)
	r_vec = positive_r - (modulus * (np.round(np.divide(positive_r, modulus).astype(float)).astype(int)))
	return r_vec

def poly_base_decomposition(f, B): 
	N = len(f)
	f = np.array(f, dtype = object)
	k = ceil( log2(2 * max(f) + 1) / log2(B))
	decomposed_f = np.zeros((N, k), dtype = object)
	for j in range(k):
		r = mod_vec(f, B)
		decomposed_f[:,j] = r 
		f = (f - r) // B
	return decomposed_f

def fast_mult(decomposed_f, decomposed_g, B = 2 ** 19):
	N = decomposed_f.shape[0]
	k_f = decomposed_f.shape[1]
	k_g = decomposed_g.shape[1]
	decomposed_fg = np.zeros((N + 1, k_f + k_g - 1), dtype = 'complex128')	

	for i in range(k_f):
		for j in range(k_g):
			decomposed_fg[:,i + j] += scipy.fft.rfft(decomposed_f[:, i], 2 * N) * scipy.fft.rfft(decomposed_g[:, j], 2 * N) 
	fg_recovered = np.array([0] * (2 * N), dtype = object)
	for j in reversed(range(k_f + k_g - 1)):
		rounded_term = scipy.fft.irfft(decomposed_fg[:,j], 2 * N)
		rounded_term = np.round(rounded_term.real).astype(int)
		fg_recovered *= B
		fg_recovered += rounded_term
	return fg_recovered	
	
def fast_poly_mult(f, g, base = 2 ** 19):
	f = np.array(f, dtype = 'object')
	g = np.array(g, dtype = 'object')
	f_dec = poly_base_decomposition(f, base)	
	g_dec = poly_base_decomposition(g, base)

	return fast_mult(f_dec, g_dec, base)


In [3]:
from random import SystemRandom
import numpy as np
from math import log2, ceil
from time import time


In [4]:


def uniform_vector(A,B): 
	'''
	Input : integers A,B
	Output: a vector of len N with uniform integer entries in range(A,B+1)  
	'''
	return [SystemRandom().randrange(B - A + 1) + A for _ in range(N)]

def round_vec(vec_x, pp, qq):
	'''
	Input : vec_x a vector of integers
	Output: nearest integer vector to vec_x * pp / qq
	'''
	vec_x = np.array(vec_x)
	return (2 * pp * vec_x + qq) // (2 * qq) 

def int2base(n, b):
	#Input : integer n and a base b
	#Output: a vector of digits corresponding to the decomposition of n in base b     
    if n < b:
        return [n]
    else:
        return [n % b] + int2base(n // b, b) 

def poly_add(p1, p2, modulus = None): 
	'''
	Input : np.arrays p1 and p2 of the same length
	Output: component-wise sum of the two vectors (reduced modulo 'modulus')
	'''
	p1 = np.array(p1, dtype = object)
	p2 = np.array(p2, dtype = object)
	addition = p1 + p2 
	if modulus == None:
		return addition
	else:	
		return np.array([x % modulus for x in addition], dtype = object)	

	'''
	Input : integer vectors p1, p2 representing the coefficients of polynomials of degree at most N - 1
	Output: integer vector representing the product polynoial p1 * p2 reduced modulo X^N + 1  
	'''  
def poly_mul(p1, p2):
    p1 = np.array(p1, dtype = object)
    p2 = np.array(p2, dtype = object)
    product = fast_poly_mult(p1, p2)
    product = np.concatenate( (product, [0] * ( 2 * N - len(product) )), None) 
    return np.array([int(product[i]) - int(product[i + N]) for i in range(N)], dtype = object)


def KeyGen():
	sk = uniform_vector(-1,1)
	a = uniform_vector(0,r - 1)
	b = round_vec(poly_mul(a, sk), q, r) % q 
	pk = (a, b)
	return (sk, pk)

def RelinKeyGen(sk, base):
	RelinKey = []
	k = ceil((2 * log2(q) - log2(p)) / log2(base)) #k = ceil(log_base(q ** 2 / p))
	ss = poly_mul(sk, sk)
	for i in range(k):
		v = uniform_vector(0, q - 1) #this is the most expensive computation in ReliNkG
		mask = round_vec(poly_mul(v, sk), p, q) % p
		w = (mask + round_vec(base ** i * ss, p ** 2, q ** 2)) % p
		RelinKey.append((v,w))
	return RelinKey

def Encrypt(message, pub_key):
	(a,b) = pub_key
	rnd = uniform_vector(-1,1)
	c0 = round_vec(poly_mul(a, rnd), q, r) % q
	c1 = round_vec(poly_mul(b, rnd), p, q) % p
	encoded_message = Delta * np.array(message, dtype = object)
	c1 = poly_add(c1, encoded_message) % p
	return (c0,c1)

def Decrypt(ct, sk):
	(c0,c1) = ct
	c0 = np.array(c0, dtype = object)
	c1 = np.array(c1, dtype = object)
	sk = np.array(sk, dtype = object)
	scaled_c0sk = -p * poly_mul(c0,sk)
	scaled_c1 = q * c1	
	return round_vec(poly_add(scaled_c1,scaled_c0sk), t, p * q) % t

def CiphertextAddition(ct1, ct2):
	return (poly_add(ct1[0], ct2[0], q), poly_add(ct1[1], ct2[1], p))

def CiphertextMultiplication(ct1, ct2, rkey):
	base = 2 
	k = ceil((2 * log2(q) - log2(p)) / log2(base)) 
	c2 = round_vec(poly_mul(ct1[1], ct2[1]), t, p) % p
	c1 = round_vec(poly_add(poly_mul(ct1[0], ct2[1]), poly_mul(ct1[1], ct2[0])), t, p) % q
	c0 = round_vec(poly_mul(ct1[0], ct2[0]), t, p) % (q ** 2 // p)

	decomposed_c0 = np.zeros((N, k), dtype = object)	
	for i in range(N):
		into_base = int2base(c0[i],base)
		decomposed_c0[i] = np.array(into_base + [0] * (k - len(into_base)), dtype = object)

	v = c1
	w = c2 
	for j in range(k):

		v = (v + poly_mul(rkey[j][0], decomposed_c0[:,j])) % q
		w = (w + poly_mul(rkey[j][1], decomposed_c0[:,j])) % p

	return (v,w)	

def noise(ct, sk, msg): 
	'''
	Input : ciphertext ct, secret key sk and the message m that ct decrypts to: i.e. Decrypt(ct, sk) == m;
	Output: the function returns pq * noise, where |noise| << 0.5; noise is actually the decryption noise and is of the form = integer / (p * q) 
			More precisely, the output is computed based on the decryption equation t/p * (-p/q * ct[0] * s + ct[1]) = msg + noise

	This function is used in noise_LPR.py script to compute the actual noise values in ciphertexts compared to the theoretical bounds from the paper.		
	'''
	(c0,c1) = ct
	msg = np.array(msg, dtype = object)
	c0 = np.array(c0, dtype = object)
	c1 = np.array(c1, dtype = object)
	if np.array_equal(msg, Decrypt((c0,c1), sk)):
		pqnoise = poly_add(q * t * c1, - p * t * poly_mul(c0,sk))
		pqnoise = poly_add(pqnoise, - p * q * msg)		
		pqnoise = mod_vec(pqnoise, p * q)
		return np.array(pqnoise, dtype = object)	
	else:
		print("Noise is too large! ct does not decrypt correctly!")


### Test for FHE

In [5]:
# import math
# parameter_sets = [
#     {"N": 2**10, "r": 2**26,  "q": 2**22,  "p": 2**18},
#     {"N": 2**11, "r": 2**52,  "q": 2**48,  "p": 2**44},
#     {"N": 2**12, "r": 2**105, "q": 2**101, "p": 2**97},
#     {"N": 2**13, "r": 2**211, "q": 2**207, "p": 2**203},
  
# ]
# attempts = 4

# for params in parameter_sets:
#     N = params["N"]
#     r = params["r"]
#     q = params["q"]
#     p = params["p"]
    
#     print("testing for parameter sets N ={}, log2(r)={}, log2(q)={}, log2(p)={}".format(N,math.log2(r),math.log2(q),math.log2(p)))
#     t = 3
#     Delta = p // t


#     (sk, pk) = KeyGen()
#     rkey = RelinKeyGen(sk, 2)

    
#     for trail in range(attempts):
#         msg1 = np.array(uniform_vector(-t//2, t - t//2 -1), dtype = object) %t
#         ct1 = Encrypt(msg1, pk)
#         avg = 0
#         i = 0
#         while(True):
#             msg2 = np.array(uniform_vector(-t//2, t - t//2 -1), dtype = object) %t
#             ct2 = Encrypt(msg2, pk)
#             ct_multiplied = CiphertextMultiplication(ct1, ct2, rkey)
#             mesg_mul = poly_mul(msg1, msg2) %t
#             if not(np.array_equal(Decrypt(ct_multiplied, sk), mesg_mul)):
#                 print("failure")
#                 break
#             print("multiplicative depth: {}".format(i))
#             i+=1
#             msg1 = mesg_mul
#             ct1 = ct_multiplied
#         avg+=i
#         print("depth: {}".format(i))
#     print("The average depth for this parameter set: {}".format(avg))
#     print("------------------------------------------------------------")

## MKFHE from RLWR (Construcrtion 1)

In [6]:
def get_error(del_r, k,d,l=10, s_i=1, w=2, security_level=128, modulo_diff_power=4):
    
    """
    Input:  - del_r: the minimum degree of the polynomial.
            - k: the number of the parties.
            - d: the circuit depth to be evaluated.
            - s_i: the norm of the secret for one party.
            - l: the log2 of the moduli to start from.
            - w : the decomposition base.
            - modulo_diff_power: the difference between powers (in our case 4) 
    Output: the parameter set that matches the required level of security.
    """
    s_norm = k*s_i
    p2byp1 = 2**(-1*modulo_diff_power)
   
    l = l-(modulo_diff_power*2)
    #(base decomposition of p_2 with base 2)
    y_in = (p2byp1)*(del_r*s_norm) + 1/2
    c_1 = (del_r*s_norm + 4)*(del_r*t) + del_r/2
    c_2= (t**2)*del_r*(del_r*s_norm/2 + 2.5) + ((p2byp1**2)*(del_r**2 )*(s_norm**2)+ (p2byp1*del_r*s_norm) + 1)/2 + (l*w*k*(del_r**2)*s_norm*p2byp1)/2 + k*l*w*del_r/2
    d_e1 = (c_1**(d-1))*(c_1*y_in + c_2)
    return round(math.log2(d_e1/k)) ### for LWR we need (2^{lambda} *(circuit_error))/(2**modulo_diff)<= (p2/t)

In [7]:
def KeyGen_multiparty(k):
    a = np.array(uniform_vector(0,r - 1), dtype=object)
    pk_list = []
    sk_list = []
    for i in range(k):
        sk = np.array(uniform_vector(-1,1), dtype=object)
        # print("sk len: ", len(sk))
        b = round_vec(poly_mul(a, sk), q, r) % q 
        pk = (a, b)
        
        sk_list.append(sk)
        pk_list.append(pk)
    
    return (sk_list, pk_list)

def KeyExt(pk_list):
    a = pk_list[0][0]
    b = pk_list[0][1]

    for i in range (1, len(pk_list)):
        b = (b + pk_list[i][1]) % q

    return (a, b)
            
            
def RelinKeyGen_multiparty(sk_list):
    base = 2
    k_parties = len(sk_list)

    l = ceil(log2(r) / log2(base)) 
    
    a_r = [np.array(uniform_vector(0, r - 1), dtype=object) for _ in range(l)]
    w_vec = [base ** j for j in range(l)]
    
    u_list = []
    h0_i_list = []
    h1_i_list = []
    
    # Step 1
    for i in range(k_parties):
        s_i = sk_list[i]

        # print("s_i: ", s_i)
        # print("s_i len: ", len(s_i))
        
        u_i = np.array(uniform_vector(-1, 1), dtype=object)
        u_list.append(u_i)
        
        h0_i = []
        h1_i = []        
        
        for j in range(l):
            term1_0 = round_vec(poly_mul(u_i, a_r[j]), q, r) % q

            term2_0 = round_vec(s_i * w_vec[j], p, q) 
            # term2_0 = ((p/q)*s_i * w_vec[j]) % p
            
            h0_i.append((term1_0 + term2_0) % q)
            
            term1_1 = round_vec(poly_mul(s_i, a_r[j]), q, r) % q
            h1_i.append(term1_1) 
            
        h0_i_list.append(h0_i)
        h1_i_list.append(h1_i)
        
    h0 = np.sum(h0_i_list, axis=0) % q
    h1 = np.sum(h1_i_list, axis=0) % q

    # print(h0)
    
    # Step 2
    hp0_i_list = []
    hp1_i_list = []
    
    for i in range(k_parties):
        s_i = sk_list[i]
        u_i = u_list[i]
        
        s_minus_u = s_i - u_i 
        
        hp0_i = []
        hp1_i = []
        
        for j in range(l):
            hp0_val = round_vec(poly_mul(s_i, h0[j]), p, q)
            hp0_i.append(hp0_val % p)
            
            hp1_val = round_vec(poly_mul(s_minus_u, h1[j]), p, q)
            hp1_i.append(hp1_val % p)
            
        hp0_i_list.append(hp0_i)
        hp1_i_list.append(hp1_i)
        
    hp0 = np.sum(hp0_i_list, axis=0) % p
    hp1 = np.sum(hp1_i_list, axis=0) % p
    
    r0 = (hp0 + hp1) % p
    r1 = h1
    
    # return (r0, r1)
    return list(zip(r0, r1))
    

def Encrypt_multiparty(message, epk):
    (a, b) = epk
    rnd = uniform_vector(-1, 1)
    c0 = round_vec(poly_mul(a, rnd), q, r) % q
    c1 = round_vec(poly_mul(b, rnd), p, q) % p
    
    encoded_message = Delta * np.array(message, dtype=object)
    c1 = (c1 + encoded_message) % p
    return (c0, c1)


def Decrypt_multiparty(sk_list, ct, lamda=128): ###lamda is passed with the circuit error 
    (c0, c1) = ct
    k_parties = len(sk_list)
    
    # p_i_list = []
    lamda = lamda -int(math.log2(k_parties)) ### divide by k to account the smudging error per each party
    sum_p_i = np.array(uniform_vector(0, 0), dtype = object)
    
    # Partial Decryption
    for i in range(k_parties):
        s_i = sk_list[i]
        e_sm = np.array(uniform_vector(0, 2**lamda), dtype = object) 
        
        mul_term = poly_mul(c0, s_i) % q       
        p_i = (mul_term + e_sm) % q
        sum_p_i=(sum_p_i+p_i) % q
        
    # Final Decryption
    
    scaled_sum = round_vec(sum_p_i, p, q)
    noisy_m = (c1 - scaled_sum) % p
    m = round_vec(noisy_m, t, p) % t
    
    return m

def CiphertextMultiplication_multiparty(ct1, ct2, rkey):
    base = 2 
    l = ceil(log2(r) / log2(base))
    
    c2 = round_vec(poly_mul(ct1[0], ct2[0]), t, p) % r
    c1 = round_vec((poly_mul(ct1[0], ct2[1]) + poly_mul(ct1[1], ct2[0])), t, p) % q
    c0 = round_vec(poly_mul(ct1[1], ct2[1]), t, p) % p
    
    decomposed_c2 = np.zeros((N, l), dtype = object)	
    for i in range(N):
        into_base = int2base(c2[i],base)
        decomposed_c2[i] = np.array(into_base + [0] * (l - len(into_base)), dtype = object)
    
    v = c0
    w = c1 
    for j in range(l):
        v = (v + poly_mul(rkey[j][0], decomposed_c2[:,j])) % p
        w = (w + poly_mul(rkey[j][1], decomposed_c2[:,j])) % q
    
    return (w,v)	



## Testing

In [8]:
# N = 2 ** 13
# r = 2 ** 206
# q = 2 ** 202
# p = 2 ** 198

# t = 3

# Delta = p // t

# k_parties = 2

# (sk_list, pk_list) = KeyGen_multiparty(k_parties)
# epk = KeyExt(pk_list)

# rkey = RelinKeyGen_multiparty(sk_list)

# msg1 = np.array(uniform_vector(-1,1), dtype=object) % t
# ct1 = Encrypt_multiparty(msg1, epk)
# dec_msg1 = Decrypt_multiparty(sk_list, ct1, lamda=128)

# print(msg1)
# print(dec_msg1)
# print(np.array_equal(msg1 , dec_msg1))

In [9]:
# msg2 = np.array(uniform_vector(-1,1), dtype=object) % t
# ct2 = Encrypt_multiparty(msg2, epk)

# print(msg1)
# print(msg2)

# add_msg = (msg1+msg2)%t
# add_ct = CiphertextAddition(ct1,ct2)
# dec_add = (Decrypt_multiparty(sk_list, add_ct,lamda=128)) %t

# print(add_msg)
# print(dec_add)
# print(np.array_equal(add_msg, dec_add))

# mul_msg = poly_mul(msg1, msg2) % t
# mul_ct = CiphertextMultiplication_multiparty(ct1, ct2, rkey)
# dec_mul = (Decrypt_multiparty(sk_list, mul_ct,lamda=128)) % t

# print(mul_msg)
# print(dec_mul)
# print(np.array_equal(mul_msg , dec_mul))

### Testing for some parameter sets from our paper (checking the actual circuit depth matches those selected in the parametersets)

In [ ]:
parameter_sets = [
{"N": 2**13, "r": 2**206,  "q": 2**202,  "p": 2**198},
{"N": 2**14, "r": 2**425, "q": 2**421, "p": 2**417},
{"N": 2**15, "r": 2**868, "q": 2**864, "p": 2**860}, 
{"N": 2**16, "r": 2**1743, "q": 2**1739, "p": 2**1735},
]

attempts = 10
k_parties = 2

for params in parameter_sets:
    N = params["N"]
    r = params["r"]
    q = params["q"]
    p = params["p"]
    
    print("testing for parameter sets N ={}, log2(r)={}, log2(q)={}, log2(p)={}".format(
        N, log2(r), log2(q), log2(p)))
    
    t = 3
    Delta = p // t

    (sk_list, pk_list) = KeyGen_multiparty(k_parties)
    
    epk = KeyExt(pk_list)
    
    rkey = RelinKeyGen_multiparty(sk_list)

    avg = 0
    
    for trial in range(attempts):
        msg1 = np.array(uniform_vector(-1,1), dtype=object) % t
        ct1 = Encrypt_multiparty(msg1, epk)
        
        i = 0
        while(True):
            msg2 = np.array(uniform_vector(-1,1), dtype=object) % t
            ct2 = Encrypt_multiparty(msg2, epk)
            
            ct_multiplied = CiphertextMultiplication_multiparty(ct1, ct2, rkey)
            
            mesg_mul = poly_mul(msg1, msg2) % t
            
            error_magnitude = get_error(N, k_parties,(i+1),math.log2(r), s_i=1, w=2, security_level=128, modulo_diff_power=4)
            decrypted_msg = Decrypt_multiparty(sk_list, ct_multiplied,lamda=128+error_magnitude)

            print("mesg_mul: ", mesg_mul)
            print("decrypted_msg: ", decrypted_msg)
            
            if not(np.array_equal(decrypted_msg, mesg_mul)):
                print("failure") 
                break
                
            print("multiplicative depth: {}".format(i))
            i += 1
            msg1 = mesg_mul
            ct1 = ct_multiplied
            
        avg += i
        print("depth: {}".format(i))
        
    print("The average depth for this parameter set: {}".format(avg / attempts)) 
    print("------------------------------------------------------------")

testing for parameter sets N =8192, log2(r)=206.0, log2(q)=202.0, log2(p)=198.0


/tmp/ipykernel_8063/1817505603.py:6: DeprecationWarning: non-integer arguments to randrange() have been deprecated since Python 3.10 and will be removed in a subsequent version
  return [SystemRandom().randrange(B - A + 1) + A for _ in range(N)]


mesg_mul:  [2 2 1 ... 1 1 0]
decrypted_msg:  [2 2 1 ... 1 1 0]
multiplicative depth: 0
mesg_mul:  [1 0 1 ... 2 2 0]
decrypted_msg:  [1 0 1 ... 2 2 0]
multiplicative depth: 1
mesg_mul:  [2 2 1 ... 2 1 2]
decrypted_msg:  [1 0 2 ... 1 0 2]
failure
depth: 2
mesg_mul:  [1 0 2 ... 0 2 2]
decrypted_msg:  [1 0 2 ... 0 2 2]
multiplicative depth: 0
mesg_mul:  [1 2 1 ... 0 1 0]
decrypted_msg:  [1 2 1 ... 0 1 0]
multiplicative depth: 1
mesg_mul:  [0 2 2 ... 1 0 0]
decrypted_msg:  [2 0 1 ... 2 0 2]
failure
depth: 2
mesg_mul:  [2 1 1 ... 1 1 1]
decrypted_msg:  [2 1 1 ... 1 1 1]
multiplicative depth: 0
mesg_mul:  [0 2 0 ... 0 0 1]
decrypted_msg:  [0 2 0 ... 0 0 1]
multiplicative depth: 1
mesg_mul:  [2 2 0 ... 0 2 1]
decrypted_msg:  [2 1 2 ... 0 1 1]
failure
depth: 2
mesg_mul:  [2 2 2 ... 2 1 1]
decrypted_msg:  [2 2 2 ... 2 1 1]
multiplicative depth: 0
mesg_mul:  [2 1 0 ... 1 2 1]
decrypted_msg:  [2 1 0 ... 1 2 1]
multiplicative depth: 1
mesg_mul:  [2 2 2 ... 1 0 1]
decrypted_msg:  [1 2 0 ... 1 0 0]
f